# CO2 Emission by Countries — Analiza i predikcija

Seminarski rad — Skladištenje podataka i otkrivanje znanja

Dataset: [CO2 Emission by Countries Year Wise (1750-2022)](https://www.kaggle.com/datasets/moazzimalibhatti/co2-emission-by-countries-year-wise-17502022)

Baznano na notebook-u: [Global Cumulative CO2 Emission Gap Analysis](https://www.kaggle.com/code/sasakitetsuya/global-cumulative-co2-emission-gap-analysis)


## 1. Učitavanje i priprema podataka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('CO2_emission_by_countries.csv', encoding='latin1')
df.columns = ['Country','Code','CallingCode','Year','CumEmission','Population','Area','PctWorld','Density']

df['PctWorld'] = df['PctWorld'].astype(str).str.replace('%','').str.replace(',','').astype(float)
df['Density'] = df['Density'].astype(str).str.replace('/km²','').str.replace(',','').astype(float)

df = df.sort_values(['Country','Year'])
df['AnnualEmission'] = df.groupby('Country')['CumEmission'].diff()

def era(y):
    if y < 1800: return 'Predindustrijski'
    elif y < 1950: return 'Industrijski'
    else: return 'Moderni'
df['Era'] = df['Year'].apply(era)

print(df.shape)
df.head(10)

## 2. Istraživačka analiza podataka (EDA)

### Slika 10 – Korelaciona matrica ključnih atributa

In [ ]:
cols = ['Year','AnnualEmission','CumEmission','Population','Area','Density','PctWorld']
corr = df[cols].corr()
print(corr.round(2))

plt.figure(figsize=(8,6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Korelaciona matrica ključnih atributa")
plt.tight_layout()
plt.savefig('slika10_korelacija.png', dpi=150)
plt.show()

### Slika 11 – Distribucija godišnje emisije CO2 (log skala)

In [ ]:
positive = df[df['AnnualEmission'] > 0]['AnnualEmission']
print('Ukupno zapisa:', len(df))
print('Zapisa sa AnnualEmission = 0:', (df['AnnualEmission']==0).sum())
print('Zapisa sa AnnualEmission > 0:', (df['AnnualEmission']>0).sum())
print(positive.describe())

plt.figure(figsize=(8,5))
sns.histplot(np.log10(positive), bins=50, color='teal')
plt.xlabel('log10(Godišnja emisija CO2, tone)')
plt.ylabel('Broj zapisa')
plt.title('Distribucija godišnje emisije CO2 (log skala)')
plt.tight_layout()
plt.savefig('slika11_distribucija.png', dpi=150)
plt.show()

### Slika 12 – Distribucija godišnje emisije CO2 po istorijskim erama

In [ ]:
positive_df = df[df['AnnualEmission'] > 0].copy()
positive_df['logEmission'] = np.log10(positive_df['AnnualEmission'])

print(positive_df.groupby('Era')['AnnualEmission'].agg(['median','count']))

plt.figure(figsize=(7,5))
order = ['Predindustrijski','Industrijski','Moderni']
sns.boxplot(data=positive_df, x='Era', y='logEmission', order=order, hue='Era', palette='Set2', legend=False)
plt.ylabel('log10(Godišnja emisija CO2, tone)')
plt.xlabel('Istorijska era')
plt.title('Distribucija godišnje emisije CO2 po istorijskim erama')
plt.tight_layout()
plt.savefig('slika12_boxplot_era.png', dpi=150)
plt.show()

### Slika 13 – Populacija vs. Godišnja emisija CO2 (2020)

In [ ]:
d2020 = df[(df['Year']==2020) & (df['AnnualEmission']>0) & (df['Population']>0)].copy()

plt.figure(figsize=(7,6))
plt.scatter(np.log10(d2020['Population']), np.log10(d2020['AnnualEmission']), alpha=0.6, color='darkorange', edgecolor='k', linewidth=0.3)
plt.xlabel('log10(Populacija, 2022)')
plt.ylabel('log10(Godišnja emisija CO2, 2020, tone)')
plt.title('Populacija vs. Godišnja emisija CO2 (2020)')
plt.tight_layout()
plt.savefig('slika13_scatter_populacija.png', dpi=150)
plt.show()

corr_2020 = np.corrcoef(np.log10(d2020['Population']), np.log10(d2020['AnnualEmission']))[0,1]
print('Broj zemalja u 2020 scatter:', len(d2020))
print('Korelacija log-log:', round(corr_2020,3))

## 3. Priprema podataka za model i podela na trening/test skup

Odabrana je regresija — cilj je predikcija godišnje emisije CO2 (`AnnualEmission`) na osnovu godine, populacije, površine i gustine naseljenosti. Atribut `PctWorld` je isključen zbog savršene korelacije sa `Area` (Slika 10). Zbog izražene asimetrije cilja (Slika 11), primenjena je log1p transformacija.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

clean = df.dropna(subset=['AnnualEmission','Population','Area','Density']).copy()
clean['LogEmission'] = np.log1p(clean['AnnualEmission'])
print('Broj zapisa nakon čišćenja:', len(clean))

features = ['Year','Population','Area','Density']
X = clean[features]
y = clean['LogEmission']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Trening skup:', len(X_train), '| Test skup:', len(X_test))

## 4. Trening i validacija modela (5-fold cross-validacija)

In [ ]:
lin_model = LinearRegression()
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

lin_cv = cross_val_score(lin_model, X_train, y_train, cv=5, scoring='r2')
rf_cv = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2')

print('Linear Regression CV R2:', lin_cv.round(3), '| Mean:', round(lin_cv.mean(),3), '| Std:', round(lin_cv.std(),3))
print('Random Forest CV R2:', rf_cv.round(3), '| Mean:', round(rf_cv.mean(),3), '| Std:', round(rf_cv.std(),3))

lin_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
print('Modeli istrenirani.')

## 5. Evaluacija modela na test skupu

In [ ]:
lin_pred = lin_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{name}: RMSE={rmse:.3f} | MAE={mae:.3f} | R2={r2:.3f}')
    return rmse, mae, r2

print('--- Rezultati na TEST skupu (log skala) ---')
lin_rmse, lin_mae, lin_r2 = evaluate('Linear Regression', y_test, lin_pred)
rf_rmse, rf_mae, rf_r2 = evaluate('Random Forest', y_test, rf_pred)

### Slika 14 – Važnost atributa u Random Forest modelu

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
print(importances)

plt.figure(figsize=(7,4))
importances.plot(kind='barh', color='darkgreen')
plt.xlabel('Važnost atributa')
plt.title('Važnost atributa u Random Forest modelu')
plt.tight_layout()
plt.savefig('slika14_feature_importance.png', dpi=150)
plt.show()

### Slika 15 – Stvarne vs. Predviđene vrednosti — poređenje modela

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,6))

axes[0].scatter(y_test, lin_pred, alpha=0.3, s=10, color='indianred')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', linewidth=2)
axes[0].set_xlabel('Stvarna vrednost (log skala)')
axes[0].set_ylabel('Predviđena vrednost (log skala)')
axes[0].set_title('Linear Regression')

axes[1].scatter(y_test, rf_pred, alpha=0.3, s=10, color='steelblue')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', linewidth=2)
axes[1].set_xlabel('Stvarna vrednost (log skala)')
axes[1].set_ylabel('Predviđena vrednost (log skala)')
axes[1].set_title('Random Forest')

plt.suptitle('Stvarne vs. Predviđene vrednosti — poređenje modela')
plt.tight_layout()
plt.savefig('slika15_predicted_vs_actual_oba.png', dpi=150)
plt.show()

## 6. Poređenje algoritama

In [ ]:
results = pd.DataFrame({
    'Algoritam': ['Linear Regression', 'Random Forest'],
    'RMSE': [lin_rmse, rf_rmse],
    'MAE': [lin_mae, rf_mae],
    'R2': [lin_r2, rf_r2],
    'CV R2 mean': [lin_cv.mean(), rf_cv.mean()],
    'CV R2 std': [lin_cv.std(), rf_cv.std()]
})
results

## 7. Interpretacija nalaza

Godina (`Year`) je daleko najvažniji prediktor godišnje emisije CO2 (~64% važnosti), što potvrđuje da je vremenski trend dominantan faktor — emisija CO2 je pre svega fenomen 20. i 21. veka, a ne funkcija same veličine ili gustine naseljenosti zemlje. Random Forest model (R²=0.992) značajno nadmašuje Linear Regression (R²=0.553), što potvrđuje da je odnos između atributa i emisije izrazito nelinearan.